In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
import re

# 1. Загрузка
df = pd.read_csv('news.csv')

# 2. Предобработка (берём заголовок + первые 200 символов текста, чтобы не размывать смысл)
def prepare_text(row):
    title = str(row['title']) if pd.notna(row['title']) else ''
    text = str(row['text'])[:200] if pd.notna(row['text']) else ''
    combined = (title + ' ' + text).lower()
    combined = re.sub(r'[^\w\sа-яё]', ' ', combined)
    return re.sub(r'\s+', ' ', combined).strip()

df['clean'] = df.apply(prepare_text, axis=1)

# 3. Векторизация
vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(df['clean'])

# 4. Матрица сходства -> матрица расстояний
sim_matrix = cosine_similarity(tfidf_matrix)
dist_matrix = np.maximum(0, 1 - sim_matrix)

# 5. Кластеризация
# eps = 0.25 означает, что группируем новости с косинусным сходством > 0.75
# min_samples=2 означает, что уникальные новости останутся шумом (-1), что нормально
clustering = DBSCAN(eps=0.25, min_samples=2, metric='precomputed')
df['cluster'] = clustering.fit_predict(dist_matrix)

# 6. Расчет метрик цитируемости
def calculate_citability(df):
    results = []
    for cluster_id in df['cluster'].unique():
        if cluster_id == -1:
            continue

        group = df[df['cluster'] == cluster_id]

        # Извлекаем домены из ссылок
        sources = group['source'].dropna().apply(
            lambda x: x.split('/')[2].replace('www.', '') if '//' in x else x
        ).unique()

        results.append({
            'cluster_id': int(cluster_id),
            'news_count': len(group),
            'unique_sources': len(sources),
            'sources': ', '.join(sources),
            'representative_title': group.iloc[0]['title'],
            'citability_score': len(group) * len(sources)
        })

    return pd.DataFrame(results).sort_values('citability_score', ascending=False)

metrics_df = calculate_citability(df)

# 7. Вывод результатов
print(f"Всего новостей: {len(df)}")
print(f"Найдено уникальных событий (кластеров): {len(metrics_df)}")
print(f"Новостей-одиночек (шум): {(df['cluster'] == -1).sum()}")
print("\n Топ-10 самых цитируемых событий:")
print(metrics_df.head(10)[['representative_title', 'news_count', 'unique_sources', 'citability_score']].to_string(index=False))
metrics_df.to_csv('citability_results.csv', index=False, encoding='utf-8')
df[['title', 'text', 'source', 'cluster']].to_csv('news_with_clusters.csv', index=False, encoding='utf-8')
print("\n Файлы citability_results.csv и news_with_clusters.csv сохранены.")

Всего новостей: 1880
Найдено уникальных событий (кластеров): 77
Новостей-одиночек (шум): 1660

 Топ-10 самых цитируемых событий:
                                                                                                                                                                                                                      representative_title  news_count  unique_sources  citability_score
                                 Четырнадцать новых методов лечения вошли в базовую программу ОМС в 2026 году, сообщил в интервью РИА Новости председатель Федерального фонда обязательного медицинского страхования (ФФОМС) Илья Баланин.          12              10               120
                        Средний размер пенсионного обеспечения более 30 тысяч рублей в мае 2026 года среди неработающих граждан отмечен в 13 регионах РФ, следует из данных Социального фонда России, с которыми ознакомилось РИА Новости.           7               7                49
                            